<a href="https://colab.research.google.com/github/eduzegarra/grade_01/blob/main/info_05_modelos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import pandas as pd
os.chdir('/content/drive/MyDrive/a_SENAMHI/OUT/')
!ls


base_final.csv	     comercio.dta    informacion.dta  top_regions.csv
caracteristicas.dta  coord_prod.csv  mq_eq.dta	      vbpagri.dta
caratulas.dta	     costos.dta      sf.dta
cofert.dta	     edu.dta	     supag.dta


In [3]:
df = pd.read_csv('base_final.csv', low_memory=False)

In [4]:
### nos quedamos con observaciones validas para info_5 y t_vbpagri

df_val=df.loc[(df.info_5.notna()) & (df.t_vbpagri.notna()) ].copy()

In [5]:
columns_to_fill = df_val.filter(regex=r'maq|equ|t_comprador|t_destino').columns
df_val[columns_to_fill] = df_val[columns_to_fill].fillna(0)

In [6]:
df_val['t_ingagri'] = (df_val.t_vbpagri - df_val.cost_tot).astype(float)

In [7]:
df_val[['t_ingagri', 't_vbpagri', 'info_5', 't_supcose', 't_usomaq', 't_usoequ',
               't_afectacion', 't_credito', 't_destino3']].describe().transpose()

,count,mean,std,min,25%,50%,75%,max
t_ingagri,97726.0,236.500838,8593.807876,-148632.5150,-1.445,-0.08,7.32,1066589.76
t_vbpagri,97726.0,370.327804,10389.772222,0.0000,0.000,2.56,17.24,1194294.00
info_5,97726.0,0.241983,0.428286,0.0000,0.000,0.00,0.00,1.00
t_supcose,97726.0,9.863517,158.773579,0.0003,0.430,1.17,3.50,20431.30
t_usomaq,97726.0,0.364468,0.481283,0.0000,0.000,0.00,1.00,1.00
t_usoequ,97726.0,0.527219,0.499261,0.0000,0.000,1.00,1.00,1.00
t_afectacion,97726.0,0.636565,0.480991,0.0000,0.000,1.00,1.00,1.00
t_credito,97726.0,0.122383,0.327729,0.0000,0.000,0.00,0.00,1.00
t_destino3,97726.0,0.051603,0.221226,0.0000,0.000,0.00,0.00,1.00


In [8]:
# Create dummy variables for REGION, dropping the first category to avoid multicollinearity
region_dummies = pd.get_dummies(df_val['REGION'], prefix='region', drop_first=True, dtype=float)
year_dummies = pd.get_dummies(df_val['ANIO'], prefix='year', drop_first=True, dtype=float)

# Join both region and year dummies back to our main enterprise dataframe as a list
df_val = df_val.join([region_dummies, year_dummies])


In [9]:
## Convertimos algunas variables a logaritmos

import numpy as np
ll=['ingagri', 'vbpagri', 'supcose', 'supag']
for dd in ll:
  # Using np.maximum(0, ...) to guarantee that the input to log1p is non-negative and avoid warnings
  df_val['log_'+dd] = np.log1p(df_val['t_'+dd])

/usr/local/lib/python3.13/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.13/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [10]:
## excluimos cuando ingreso neto agrario es negativo
df_val.loc[df_val.log_ingagri<0, 'log_ingagri'] = np.nan

In [11]:
df_val.filter(regex='log').describe().transpose()

,count,mean,std,min,25%,50%,75%,max
log_ingagri,47032.0,2.389077,1.689148,0.000000,1.090496,2.199444,3.380723,13.879978
log_vbpagri,97726.0,1.729344,1.806384,0.000000,0.000000,1.269761,2.903617,13.993067
log_supcose,97726.0,1.084367,1.014395,0.000300,0.357674,0.774727,1.504077,9.924872
log_supag,97726.0,1.749845,1.189516,0.001998,0.875052,1.504077,2.351375,10.757924


In [14]:
!pip install -q linearmodels
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS

# 1. Create separate DataFrames for Empresas and Productores with explicit copies
df_emp = df_val.loc[df_val['tipo'] == 'Empresa'].copy()
df_pro = df_val.loc[df_val['tipo'] == 'Productor'].copy()

# 2. Define independent variables lists
region_cols = ['region_Sierra', 'region_Selva']
year_cols_emp = ['year_2025']
year_cols_pro = ['year_2024', 'year_2025']

# Note: based on previous cell edits, the column name is t_usoequ (or t_usoequip/t_usoequ depending on data)
# Let's dynamically find the correct usage column name for equipment
# usoequ_col = 't_usoequ' if 't_usoequ' in df_val.columns else 't_usoequip'

vars_common = ['info_5', 'log_supcose', 't_usomaq', 't_usoequ',
               't_afectacion', 't_destino3'] + region_cols
vars_emp = vars_common + year_cols_emp
vars_pro = vars_common + ['t_jhedu', 't_jhedad', 't_jhmujer'] + year_cols_pro

# Clean missing values for each model subset specifically
model_vars_emp = ['log_vbpagri', 'FACTOR_PRODUCTOR', 'neces_AGROCLIM', 'log_supag'] + vars_common
base_emp = df_emp.dropna(subset=model_vars_emp).copy()

model_vars_pro = ['log_vbpagri', 'FACTOR_PRODUCTOR', 'neces_AGROCLIM', 'log_supag'] + vars_pro
base_pro = df_pro.dropna(subset=model_vars_pro).copy()

# Add constants explicitly for IV estimation
base_emp['const'] = 1.0
base_pro['const'] = 1.0

# --- EMPRESAS MODEL ---
print("=== EMPRESAS (COMPANIES) ===")
# Weighted OLS using WLS
formula_emp = f"log_vbpagri ~ {' + '.join(vars_emp)}"
wls_emp = smf.wls(formula_emp, data=base_emp,
                  weights=base_emp['FACTOR_PRODUCTOR']).fit(cov_type='HC1')
print(wls_emp.summary())

# IV-2SLS Endogeneity Test
exog_emp = ['const'] + [v for v in vars_emp if v != 'info_5']
iv_emp = IV2SLS(dependent=base_emp['log_vbpagri'],
                 exog=base_emp[exog_emp],
                 endog=base_emp['info_5'],
                 instruments=base_emp[['neces_AGROCLIM', 'log_supag']],
                 weights=base_emp['FACTOR_PRODUCTOR']).fit(cov_type='robust')
print("\n--- Endogeneity Test (Empresas) ---")
print(iv_emp.wu_hausman())


# --- PRODUCTORES MODEL ---
print("\n\n=== PRODUCTORES (INDIVIDUAL PRODUCERS) ===")
# Weighted OLS using WLS
formula_pro = f"log_vbpagri ~ {' + '.join(vars_pro)}"
wls_pro = smf.wls(formula_pro, data=base_pro,
                  weights=base_pro['FACTOR_PRODUCTOR']).fit(cov_type='HC1')
print(wls_pro.summary())

# IV-2SLS Endogeneity Test
exog_pro = ['const'] + [v for v in vars_pro if v != 'info_5']
iv_pro = IV2SLS(dependent=base_pro['log_vbpagri'],
                 exog=base_pro[exog_pro],
                 endog=base_pro['info_5'],
                 instruments=base_pro[['neces_AGROCLIM', 'log_supag']],
                 weights=base_pro['FACTOR_PRODUCTOR']).fit(cov_type='robust')
print("\n--- Endogeneity Test (Productores) ---")
print(iv_pro.wu_hausman())

=== EMPRESAS (COMPANIES) ===
                            WLS Regression Results                            
Dep. Variable:            log_vbpagri   R-squared:                       0.721
Model:                            WLS   Adj. R-squared:                  0.720
Method:                 Least Squares   F-statistic:                     1038.
Date:                Tue, 20 Oct 2026   Prob (F-statistic):               0.00
Time:                        16:38:21   Log-Likelihood:                -4690.9
No. Observations:                2227   AIC:                             9402.
Df Residuals:                    2217   BIC:                             9459.
Df Model:                           9                                         
Covariance Type:                  HC1                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept        

In [18]:
!pip install -q linearmodels
import pandas as pd
import numpy as np
from linearmodels.iv import IV2SLS

# Re-prepare datasets to ensure all variables exist in current memory
df_emp = df_val.loc[df_val['tipo'] == 'Empresa'].copy()
df_pro = df_val.loc[df_val['tipo'] == 'Productor'].copy()

# 2. Define independent variables lists
region_cols = ['region_Sierra', 'region_Selva']
year_cols_emp = ['year_2025']
year_cols_pro = ['year_2024', 'year_2025']

vars_common = ['info_5', 'log_supcose', 't_usomaq', 't_usoequ',
               't_afectacion', 't_destino3'] + region_cols
vars_emp = vars_common + year_cols_emp
vars_pro = vars_common + ['t_jhedu', 't_jhedad', 't_jhmujer'] + year_cols_pro

model_vars_emp = ['log_vbpagri', 'FACTOR_PRODUCTOR', 'neces_AGROCLIM', 'log_supag'] + vars_emp
base_emp = df_emp.dropna(subset=model_vars_emp).copy()

model_vars_pro = ['log_vbpagri', 'FACTOR_PRODUCTOR', 'neces_AGROCLIM', 'log_supag'] + vars_pro
base_pro = df_pro.dropna(subset=model_vars_pro).copy()

base_emp['const'] = 1.0
base_pro['const'] = 1.0

exog_emp = ['const'] + [v for v in vars_emp if v != 'info_5']
exog_pro = ['const'] + [v for v in vars_pro if v != 'info_5']

# --- IV-2SLS Model for Empresas ---
print("=== IV-2SLS REGRESSION RESULTS: EMPRESAS ===")
iv_emp_fit = IV2SLS(dependent=base_emp['log_vbpagri'],
                     exog=base_emp[exog_emp],
                     endog=base_emp['info_5'],
                     instruments=base_emp[['neces_AGROCLIM','log_supag']],
                     weights=base_emp['FACTOR_PRODUCTOR']).fit(cov_type='robust')
print(iv_emp_fit.summary)

# --- IV-2SLS Model for Productores ---
print("\n\n=== IV-2SLS REGRESSION RESULTS: PRODUCTORES ===")
iv_pro_fit = IV2SLS(dependent=base_pro['log_vbpagri'],
                     exog=base_pro[exog_pro],
                     endog=base_pro['info_5'],
                     instruments=base_pro['neces_AGROCLIM'],
                     weights=base_pro['FACTOR_PRODUCTOR']).fit(cov_type='robust')
print(iv_pro_fit.summary)

=== IV-2SLS REGRESSION RESULTS: EMPRESAS ===
                          IV-2SLS Estimation Summary                          
Dep. Variable:            log_vbpagri   R-squared:                      0.7106
Estimator:                    IV-2SLS   Adj. R-squared:                 0.7094
No. Observations:                2227   F-statistic:                    8661.4
Date:                Tue, Oct 20 2026   P-value (F-stat)                0.0000
Time:                        16:45:53   Distribution:                  chi2(9)
Cov. Estimator:                robust                                         
                                                                              
                               Parameter Estimates                               
               Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
---------------------------------------------------------------------------------
const             1.1073     0.1779     6.2242     0.0000      0.7586      1.

In [20]:
import numpy as np

# Coefficients from log_vbpagri IV-2SLS models
coef_emp_log = 1.2025
coef_pro_log = 0.4128

# 1. Firms (Empresas)
# Average level of production (t_vbpagri) for firms who received info (info_5 == 1)
mean_vbp_emp_treated = base_emp.loc[base_emp['info_5'] == 1, 't_vbpagri'].mean()
# Total expanded number of firm users
weighted_users_emp = base_emp.loc[base_emp['info_5'] == 1, 'FACTOR_PRODUCTOR'].sum()
# Absolute average impact per treated firm (in thousands of Soles)
# For log-linear coefficient beta, the absolute change is approximately beta * mean_y
abs_impact_emp = coef_emp_log * mean_vbp_emp_treated
total_impact_emp_thousands = abs_impact_emp * weighted_users_emp
annual_impact_emp_soles = (total_impact_emp_thousands * 1000) / 2

# 2. Producers (Productores)
mean_vbp_pro_treated = base_pro.loc[base_pro['info_5'] == 1, 't_vbpagri'].mean()
weighted_users_pro = base_pro.loc[base_pro['info_5'] == 1, 'FACTOR_PRODUCTOR'].sum()
abs_impact_pro = coef_pro_log * mean_vbp_pro_treated
total_impact_pro_thousands = abs_impact_pro * weighted_users_pro
annual_impact_pro_soles = (total_impact_pro_thousands * 1000) / 3

print("=== EXPANDED ANNUAL IMPACT ON PRODUCTION VALUE (LOG SPECIFICATION) ===\n")
print(f"FIRMS (EMPRESAS - 2 Years Period):")
print(f"  - Log Coefficient: {coef_emp_log:.4f}")
print(f"  - Mean Production of Treated: S/ {mean_vbp_emp_treated * 1000:,.2f}")
print(f"  - Expanded Info Users: {weighted_users_emp:,.2f}")
print(f"  - ANNUAL IMPACT: S/ {annual_impact_emp_soles:,.2f}\n")

print(f"PRODUCERS (PRODUCTORES - 3 Years Period):")
print(f"  - Log Coefficient: {coef_pro_log:.4f}")
print(f"  - Mean Production of Treated: S/ {mean_vbp_pro_treated * 1000:,.2f}")
print(f"  - Expanded Info Users: {weighted_users_pro:,.2f}")
print(f"  - ANNUAL IMPACT: S/ {annual_impact_pro_soles:,.2f}")

=== EXPANDED ANNUAL IMPACT ON PRODUCTION VALUE (LOG SPECIFICATION) ===

FIRMS (EMPRESAS - 2 Years Period):
  - Log Coefficient: 1.2025
  - Mean Production of Treated: S/ 23,212,996.08
  - Expanded Info Users: 1,136.00
  - ANNUAL IMPACT: S/ 15,854,940,580.83

PRODUCERS (PRODUCTORES - 3 Years Period):
  - Log Coefficient: 0.4128
  - Mean Production of Treated: S/ 22,280.36
  - Expanded Info Users: 1,894,551.94
  - ANNUAL IMPACT: S/ 5,808,274,083.07


In [22]:
import pandas as pd

# Construct comparison table data
comparison_data = {
    'Producer Segment': [
        'Firms (Empresas)',
        'Firms (Empresas)',
        'Producers (Productores)',
        'Producers (Productores)'
    ],
    'Estimation Method': [
        'Weighted OLS (WLS)',
        'IV-2SLS',
        'Weighted OLS (WLS)',
        'IV-2SLS'
    ],
    'info_5 Coefficient': [
        0.3383,
        1.2025,
        0.0573,
        0.4128
    ],
    'Std. Error': [
        0.096,
        0.4614,
        0.019,
        0.1367
    ],
    'P-value': [
        '< 0.001',
        '0.0092',
        '0.003',
        '0.0025'
    ],
    'Dependent Variable': [
        'log_vbpagri',
        'log_vbpagri',
        'log_vbpagri',
        'log_vbpagri'
    ]
}

# Create DataFrame
df_comp = pd.DataFrame(comparison_data)
display(df_comp)

,Producer Segment,Estimation Method,info_5 Coefficient,Std. Error,P-value,Dependent Variable
0,Firms (Empresas),Weighted OLS (WLS),0.3383,0.0960,< 0.001,log_vbpagri
1,Firms (Empresas),IV-2SLS,1.2025,0.4614,0.0092,log_vbpagri
2,Producers (Productores),Weighted OLS (WLS),0.0573,0.0190,0.003,log_vbpagri
3,Producers (Productores),IV-2SLS,0.4128,0.1367,0.0025,log_vbpagri


In [23]:
import pandas as pd

# Defining variables list in order
variables_order = [
    'info_5', 'log_supcose', 't_usomaq', 't_usoequ', 't_afectacion', 't_destino3',
    'region_Sierra', 'region_Selva', 't_jhedu', 't_jhedad', 't_jhmujer',
    'year_2024', 'year_2025', 'Intercept/const'
]

# Creating the comparison structure
comparison_results = []

# --- WLS EMPRESAS ---
# From the printed output in cell Eof3BfZp3gvn
wls_emp_data = {
    'Model': 'Firms: WLS',
    'N': 2227,
    'R2': 0.721,
    'info_5': '0.3383*** (0.096)',
    'log_supcose': '0.8524*** (0.041)',
    't_usomaq': '0.2750** (0.107)',
    't_usoequ': '-0.0167 (0.110)',
    't_afectacion': '0.2321*** (0.087)',
    't_destino3': '2.7883*** (0.146)',
    'region_Sierra': '-2.2813*** (0.143)',
    'region_Selva': '-1.3422*** (0.169)',
    't_jhedu': '-',
    't_jhedad': '-',
    't_jhmujer': '-',
    'year_2024': '-',
    'year_2025': '0.0758 (0.087)',
    'Intercept/const': '1.1825*** (0.168)'
}

# --- IV-2SLS EMPRESAS ---
# From output in cell 7fdeec07
iv_emp_data = {
    'Model': 'Firms: IV-2SLS',
    'N': 2227,
    'R2': 0.7106,
    'info_5': '1.2025*** (0.4614)',
    'log_supcose': '0.8158*** (0.0442)',
    't_usomaq': '0.1685 (0.1248)',
    't_usoequ': '-0.1056 (0.1212)',
    't_afectacion': '0.1706* (0.0929)',
    't_destino3': '2.5979*** (0.1807)',
    'region_Sierra': '-2.2921*** (0.1451)',
    'region_Selva': '-1.2496*** (0.1740)',
    't_jhedu': '-',
    't_jhedad': '-',
    't_jhmujer': '-',
    'year_2024': '-',
    'year_2025': '0.1095 (0.0904)',
    'Intercept/const': '1.1073*** (0.1779)'
}

# --- WLS PRODUCTORES ---
# From output in cell Eof3BfZp3gvn
wls_pro_data = {
    'Model': 'Producers: WLS',
    'N': 95499,
    'R2': 0.449,
    'info_5': '0.0573*** (0.019)',
    'log_supcose': '0.8473*** (0.023)',
    't_usomaq': '0.1868*** (0.024)',
    't_usoequ': '0.2686*** (0.021)',
    't_afectacion': '-0.1832*** (0.022)',
    't_destino3': '0.9506*** (0.036)',
    'region_Sierra': '-1.1618*** (0.032)',
    'region_Selva': '-0.3570*** (0.034)',
    't_jhedu': '0.0304*** (0.006)',
    't_jhedad': '-0.0044*** (0.001)',
    't_jhmujer': '-0.1175*** (0.021)',
    'year_2024': '-0.0151 (0.023)',
    'year_2025': '-0.0174 (0.024)',
    'Intercept/const': '1.7291*** (0.063)'
}

# --- IV-2SLS PRODUCTORES ---
# From output in cell 7fdeec07
iv_pro_data = {
    'Model': 'Producers: IV-2SLS',
    'N': 95499,
    'R2': 0.4357,
    'info_5': '0.4128*** (0.1367)',
    'log_supcose': '0.8654*** (0.0231)',
    't_usomaq': '0.0956** (0.0383)',
    't_usoequ': '0.2595*** (0.0227)',
    't_afectacion': '-0.2190*** (0.0256)',
    't_destino3': '0.9284*** (0.0352)',
    'region_Sierra': '-1.2444*** (0.0425)',
    'region_Selva': '-0.3976*** (0.0359)',
    't_jhedu': '0.0250*** (0.0066)',
    't_jhedad': '-0.0048*** (0.0009)',
    't_jhmujer': '-0.1217*** (0.0225)',
    'year_2024': '-0.0251 (0.0252)',
    'year_2025': '-0.0168 (0.0253)',
    'Intercept/const': '1.6582*** (0.0677)'
}

# Combine and format table
df_all_models = pd.DataFrame([wls_emp_data, iv_emp_data, wls_pro_data, iv_pro_data])
columns_ordered = ['Model', 'N', 'R2'] + variables_order
df_all_models = df_all_models[columns_ordered]

display(df_all_models.T)

,0,1,2,3
Model,Firms: WLS,Firms: IV-2SLS,Producers: WLS,Producers: IV-2SLS
N,2227,2227,95499,95499
R2,0.721,0.7106,0.449,0.4357
info_5,0.3383*** (0.096),1.2025*** (0.4614),0.0573*** (0.019),0.4128*** (0.1367)
log_supcose,0.8524*** (0.041),0.8158*** (0.0442),0.8473*** (0.023),0.8654*** (0.0231)
t_usomaq,0.2750** (0.107),0.1685 (0.1248),0.1868*** (0.024),0.0956** (0.0383)
t_usoequ,-0.0167 (0.110),-0.1056 (0.1212),0.2686*** (0.021),0.2595*** (0.0227)
t_afectacion,0.2321*** (0.087),0.1706* (0.0929),-0.1832*** (0.022),-0.2190*** (0.0256)
t_destino3,2.7883*** (0.146),2.5979*** (0.1807),0.9506*** (0.036),0.9284*** (0.0352)
region_Sierra,-2.2813*** (0.143),-2.2921*** (0.1451),-1.1618*** (0.032),-1.2444*** (0.0425)
